# YOLOv8s 训练 - Block 检测

在 Kaggle 上训练自定义 YOLO 模型。

## 前提
1. 先用 `prepare_dataset.py` 处理好数据集 → 得到 `dataset_switch.zip`
2. 上传 `dataset_switch.zip` 到 Kaggle Dataset
3. 本 Notebook 已关联数据集
4. 右侧 Settings → Accelerator 已选 **GPU P100**

按顺序运行每个单元格即可。

In [ ]:
# 1. 安装依赖
!pip install ultralytics -q
from ultralytics import YOLO
import os

In [ ]:
# 2. 从 Kaggle Dataset 加载数据
# 左侧 Add Data → 搜索并添加你的 block-dataset
# 或者运行下面代码自动寻找

import kagglehub
import zipfile
import shutil

# 寻找数据集路径（Kaggle 会自动挂载）
import glob
zip_candidates = glob.glob('/kaggle/input/**/dataset_switch.zip', recursive=True)

if zip_candidates:
    data_path = zip_candidates[0]
    print(f'✅ 找到数据集: {data_path}')
    # 解压
    with zipfile.ZipFile(data_path, 'r') as zf:
        zf.extractall('/content/dataset')
    print('✅ 解压完成')
else:
    # 如果路径不对，列出可用的输入
    print('⚠ 未找到 dataset_switch.zip，列出 /kaggle/input/:')
    !ls -la /kaggle/input/

# 检查解压后的文件结构
if os.path.exists('/content/dataset/data.yaml'):
    train_count = len(os.listdir('/content/dataset/images/train'))
    val_count = len(os.listdir('/content/dataset/images/val'))
    print(f'📊 训练集: {train_count} 张')
    print(f'📊 验证集: {val_count} 张')
    print('✅ 数据集准备就绪！')
else:
    print('❌ data.yaml 未找到，请检查数据集路径')
    !ls -la /content/dataset/

In [ ]:
# 3. 查看数据集的 data.yaml
!cat /content/dataset/data.yaml

In [ ]:
# 4. 开始训练 YOLOv8s（block 检测）
import torch
print(f'🚀 GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU 型号: {torch.cuda.get_device_name(0)}')

model = YOLO('yolov8s.pt')

model.train(
    data='/content/dataset/data.yaml',
    epochs=200,                # 训练轮数
    imgsz=640,                 # 输入图片尺寸
    batch=16,                  # 批次大小（P100 16GB 可跑 16）
    device='cuda',
    workers=2,                 # Kaggle 限制线程数
    patience=50,               # 50 轮无提升就早停
    name='custom_block',
    exist_ok=True,
    verbose=True,
    plots=True,                # 生成训练曲线图
    save=True,                 # 自动保存 best.pt + last.pt
)

In [ ]:
# 5. 查看训练结果
print('📁 训练结果:')
!ls -lh /content/dataset/runs/detect/custom_block/weights/
print()
print('📊 训练曲线:')
!ls /content/dataset/runs/detect/custom_block/*.png

In [ ]:
# 6. 下载训练好的模型到本地
import json
from IPython.display import FileLink

best_pt = '/content/dataset/runs/detect/custom_block/weights/best.pt'
last_pt = '/content/dataset/runs/detect/custom_block/weights/last.pt'

if os.path.exists(best_pt):
    print(f'✅ best.pt 已就绪')
    # Kaggle 提供下载链接
    print('点击下方链接下载:')
    display(FileLink(best_pt))

## 意外中断怎么办？

Kaggle 有自动保存，如果中途断线：
1. 重新打开 Notebook
2. 重新运行单元格 1~3（安装 + 解压数据集）
3. 运行下面的**断点续训**单元格即可从上次保存的 last.pt 继续

**Kaggle 不会因为空闲超时断开**（Colab 会），所以比 Colab 稳定很多。

In [ ]:
# [可选] 断点续训：从中断处继续训练
# 如果训练被中断，运行这个单元格继续

last_ckpt = '/content/dataset/runs/detect/custom_scissors/weights/last.pt'

if os.path.exists(last_ckpt):
    print(f'✅ 找到断点: {last_ckpt}')
    print(f'   大小: {os.path.getsize(last_ckpt) / 1024 / 1024:.1f} MB')
    
    model = YOLO(last_ckpt)
    model.train(
        resume=True,  # ← 关键参数：自动从断点继续
    )
else:
    print('❌ 未找到断点，请先运行训练单元格')